<a href="https://colab.research.google.com/github/MaxDecub/name-preprocessing-for-record-linkage/blob/main/Names_standardization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import unicodedata
import re

In [ ]:
# ============================================================
# 1. CREATE A SAMPLE LIST OF 100 NAMES
# Alternatively, you can import the dataset containing your list of names.
# ============================================================
# Note:
# - I have retained accented characters, umlauts, apostrophes, spaces in surnames, etc.


names = [
    "José García",
    "François Dubois",
    "Søren Møller",
    "Jürgen Müller",
    "Björn Åkesson",
    "Łukasz Kowalski",
    "Dávid Szabó",
    "Tomáš Novák",
    "Marek Dvořák",
    "Oleksandr Shevchenko",
    "João Ferreira",
    "Ángel Rodríguez",
    "Zoë Van den Berg",
    "Élodie Martin",
    "Günther Schäfer",
    "Mária Nagy",
    "Iván López",
    "Anaïs Moreau",
    "Gaël Le Roux",
    "Nikola Petrović",
    "Michal Król",
    "Artūras Kazlauskas",
    "Antti Hämäläinen",
    "Lars Østergaard",
    "Bálint Tóth",
    "András Horváth",
    "Pavel Černý",
    "Jiří Šimek",
    "Đorđe Jovanović",
    "Dragana Živković",
    "Mihai Popescu",
    "Ionuț Dumitrescu",
    "Bogdan Ionescu",
    "Ștefan Georgescu",
    "Andrei Mureșan",
    "Álvaro De la Cruz",
    "Carlos Del Río",
    "María De los Santos",
    "Diego De Luca",
    "Matteo Di Stefano",
    "Luca D’Angelo",
    "Giovanni De Santis",
    "Antonio Del Vecchio",
    "Marco Di Giovanni",
    "Stefano Della Torre",
    "Pierre De Villiers",
    "Jean-Luc Morel",
    "Étienne Lévêque",
    "Lucien De La Fontaine",
    "André Dupont",
    "Thomas Van der Meer",
    "Jan Van den Broek",
    "Pieter Van Dijk",
    "Willem De Vries",
    "Koen Van der Linden",
    "Klaus Von Weber",
    "Friedrich Von Neumann",
    "Lukas Von Stein",
    "Hans-Georg Weiß",
    "Dieter Schröder",
    "Matthias Krüger",
    "Stefan Hölzer",
    "Rüdiger Kühn",
    "Wolfgang Bäcker",
    "Timo Jäger",
    "Oskar Grönberg",
    "Åke Lindström",
    "Göran Sjöberg",
    "Nils Åström",
    "Pär Björk",
    "Krzysztof Wiśniewski",
    "Paweł Zieliński",
    "Wojciech Kamiński",
    "Tomasz Piątek",
    "Grzegorz Bąk",
    "Zdeněk Beneš",
    "Václav Procházka",
    "Luboš Šťastný",
    "Radek Kučera",
    "Bohumil Němec",
    "Aleksandar Stojanović",
    "Vladimir Kovačević",
    "Milica Pavlović",
    "Katarina Nikolić",
    "Jelena Marković",
    "Igor Smirnov",
    "Dmitri Ivanov",
    "Sergei Petrov",
    "Nikolai Volkov",
    "Alexei Kuznetsov",
    "Clément De Rochefort",
    "Théo D’Aubigny",
    "René Lévesque",
    "François-Xavier Delorme",
    "Sébastien L’Écuyer",
    "Maël De Kerouac",
    "Brice D’Harcourt",
    "Yves De Montfort",
    "Pascal De La Tour",
    "Olivier D’Arcy"
]

# Creating the basing DataFrame
df = pd.DataFrame({"FULL_NAME_ORIG": names})

In [ ]:
# ============================================================
# 2. SEPARATION IN NAME AND SURNAME
# ============================================================
# Simple rule:
# - everything before the first space is considered a FIRST NAME
# - everything after is considered a LAST NAME
# This approach is useful for a demo dataset like this.
# In real-world cases, you might want to use more sophisticated rules for compound names.

df["NAME_ORIG"] = df["FULL_NAME_ORIG"].str.split(" ", n=1).str[0]
df["SURNAME_ORIG"] = df["FULL_NAME_ORIG"].str.split(" ", n=1).str[1]

In [ ]:
df

,FULL_NAME_ORIG,NAME_ORIG,SURNAME_ORIG
0,José García,José,García
1,François Dubois,François,Dubois
2,Søren Møller,Søren,Møller
3,Jürgen Müller,Jürgen,Müller
4,Björn Åkesson,Björn,Åkesson
...,...,...,...
95,Maël De Kerouac,Maël,De Kerouac
96,Brice D’Harcourt,Brice,D’Harcourt
97,Yves De Montfort,Yves,De Montfort
98,Pascal De La Tour,Pascal,De La Tour


In [ ]:
# ============================================================
# 3. STANDARDIZATION / NORMALIZATION FUNCTION
# ============================================================
# Applied rules:
# - trim leading/trailing spaces
# - explicit replacement of some special characters
# - Unicode NFKD normalization (for example, breaks down the u with two dots into two separate characters, so that the dots can then be removed and the u remains plain)
# - removal of diacritics (accents and other special characters found in many languages).
# - conversion to uppercase
# - removal/replacement of punctuation
# - normalization of multiple spaces
#
# Important note:
# - We do NOT remove accessory words such as DE / DI / VAN / VON / DEL / DE LA, etc.


def text_standardization(text):
    # Gestione dei valori null
    if pd.isna(text):
        return None

    # Converting to string
    text = str(text)

    # We remove leading/trailing spaces
    text = text.strip()

    # --------------------------------------------------------
    # EXPLICIT SPECIAL CHARACTER SUBSTITUTIONS
    # --------------------------------------------------------
    # Some characters aren't always handled well by Unicode decomposition alone,
    # so we do an explicit mapping.
    substitutions = {
        "ß": "ss",
        "ẞ": "SS",
        "ø": "o",
        "Ø": "O",
        "å": "a",
        "Å": "A",
        "æ": "ae",
        "Æ": "AE",
        "œ": "oe",
        "Œ": "OE",
        "ł": "l",
        "Ł": "L",
        "đ": "d",
        "Đ": "D",
        "ð": "d",
        "Ð": "D",
        "þ": "th",
        "Þ": "TH",
        "ı": "i",
        "İ": "I"
    }

    for special_character, substitution in substitutions.items():
        text = text.replace(special_character, substitution)

    # --------------------------------------------------------
    # UNICODE NORMALIZATION
    # --------------------------------------------------------
    # NFKD separates the base letter (e.g. the letter e from any diacritical mark, e.g. the open accent)
    text = unicodedata.normalize("NFKD", text)

    # --------------------------------------------------------
    # REMOVING DIACRITICS
    # --------------------------------------------------------
    # We only keep characters that are NOT "combining marks" (e.g., è corresponds to e with an accent. We only keep the e)
    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )

    # --------------------------------------------------------
    # STANDARDIZING APOSTROPHES AND HYPHENS
    # --------------------------------------------------------
    # Instead of immediately removing them, we're turning them into spaces
    # so we avoid joining words that were previously separate.
    text = text.replace("’", " ")
    text = text.replace("'", " ")
    text = text.replace("`", " ")
    text = text.replace("-", " ")

    # --------------------------------------------------------
    # REMOVING OTHER PUNCTUATION
    # --------------------------------------------------------
    # Anything other than a letter, number, or space is replaced with a space.
    text = re.sub(r"[^A-Za-z0-9\s]", " ", text)

    # --------------------------------------------------------
    # CAPITAL LETTERS
    # --------------------------------------------------------
    text = text.upper()

    # --------------------------------------------------------
    # SPACE NORMALIZATION
    # --------------------------------------------------------
    # Multiple consecutive spaces -> only one
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
# ============================================================
# 4. APPLICATION OF STANDARDIZATION
# ============================================================

df["NAME_STD"] = df["NAME_ORIG"].apply(text_standardization)
df["SURNAME_STD"] = df["SURNAME_ORIG"].apply(text_standardization)

# We recompose standardized name + surname
df["FULL_NAME_STD"] = (df["NAME_STD"].fillna("") + " " + df["SURNAME_STD"].fillna("")).str.strip()

In [ ]:
# ===================================================================
# 5. CREATING THE LINKAGE KEY
# ============================================================================
# Here we remove the spaces to create a simple and compact key.
# Example:
# "JURGEN MULLER" -> "JURGENMULLER"

df["LINKAGE_KEY"] = df["FULL_NAME_STD"].str.replace(" ", "", regex=False)

In [ ]:
# ============================================================
# 6. VIEW RESULT
# ============================================================

print(df[[
    "FULL_NAME_ORIG",
    "NAME_ORIG",
    "SURNAME_ORIG",
    "NAME_STD",
    "SURNAME_STD",
    "FULL_NAME_STD",
    "LINKAGE_KEY"
]].to_string(index=False))


         FULL_NAME_ORIG       NAME_ORIG   SURNAME_ORIG        NAME_STD    SURNAME_STD           FULL_NAME_STD           LINKAGE_KEY
            José García            José         García            JOSE         GARCIA             JOSE GARCIA            JOSEGARCIA
        François Dubois        François         Dubois        FRANCOIS         DUBOIS         FRANCOIS DUBOIS        FRANCOISDUBOIS
           Søren Møller           Søren         Møller           SOREN         MOLLER            SOREN MOLLER           SORENMOLLER
          Jürgen Müller          Jürgen         Müller          JURGEN         MULLER           JURGEN MULLER          JURGENMULLER
          Björn Åkesson           Björn        Åkesson           BJORN        AKESSON           BJORN AKESSON          BJORNAKESSON
        Łukasz Kowalski          Łukasz       Kowalski          LUKASZ       KOWALSKI         LUKASZ KOWALSKI        LUKASZKOWALSKI
            Dávid Szabó           Dávid          Szabó           DAVID      

In [ ]:
# ====================================================================
# 7. SAVING TO CSV FILE
# ============================================================================
# If you want to save the result:
df.to_csv("standardized_names.csv", index=False, encoding="utf-8-sig")